# VAE-GAN v3 (10K Dataset)

This notebook augments the successful Beta-VAE (v2) architecture with a PatchGAN Discriminator. 
The generator (VAE) must now satisfy the reconstruction (MSE) loss, the latent structure (KL divergence) loss, and an adversarial loss (fooling the discriminator) to cure any inherent blurriness.

In [ ]:
import os
import random
import hashlib
import json
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

BASE_DIR = "/kaggle/working/VAE_GAN_10K"
DIRS = {
    "checkpoints": os.path.join(BASE_DIR, "checkpoints"),
    "logs": os.path.join(BASE_DIR, "logs"),
    "config": os.path.join(BASE_DIR, "config")
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)


## 1. Load Dataset (10K Slice)

In [ ]:
!pip install -q datasets
from datasets import load_dataset
dataset = load_dataset("evilsocket/alucard-sprites")
train_data = dataset["train"]

def image_hash(image):
    pixels = np.asarray(image.convert("RGBA"), dtype=np.uint8)
    return hashlib.sha256(pixels.tobytes()).hexdigest()

all_hashes = []
seen_hashes = set()
for i, item in enumerate(train_data):
    h = image_hash(item["image"])
    if h not in seen_hashes:
        seen_hashes.add(h)
        all_hashes.append(i)

unique_dataset = train_data.select(all_hashes)
unique_dataset = unique_dataset.shuffle(seed=42)

TRAIN_SIZE = 9_000
VAL_SIZE = 1_000
TEST_SIZE = 1_000
TOTAL_SIZE = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

clean_dataset = unique_dataset.select(range(TOTAL_SIZE))
train_dataset = clean_dataset.select(range(0, TRAIN_SIZE))
val_dataset = clean_dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
test_dataset = clean_dataset.select(range(TRAIN_SIZE + VAL_SIZE, TOTAL_SIZE))

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

def data_generator(hf_dataset):
    for item in hf_dataset:
        image = item["image"].convert("RGBA")
        image_np = np.array(image, dtype=np.float32) / 255.0
        yield (image_np, image_np)

BATCH_SIZE = 32

def create_tf_dataset(hf_dataset):
    return tf.data.Dataset.from_generator(
        lambda: data_generator(hf_dataset),
        output_signature=(
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(128, 128, 4), dtype=tf.float32)
        )
    )

train_ds = create_tf_dataset(train_dataset).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
val_ds = create_tf_dataset(val_dataset).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
test_ds = create_tf_dataset(test_dataset).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_originals = []
for images, _ in test_ds.take(2): # load 64 images for quick eval plotting
    test_originals.append(images.numpy())
test_originals = np.concatenate(test_originals, axis=0)


## 2. VAE Generator & PatchGAN Discriminator Architecture

In [ ]:
strategy = tf.distribute.MirroredStrategy()
print("Number of devices:", strategy.num_replicas_in_sync)

with strategy.scope():
    latent_dim = 256

    # 1. Generator (VAE Encoder & Decoder)
    class Sampling(layers.Layer):
        def call(self, inputs):
            z_mean, z_log_var = inputs
            batch = tf.shape(z_mean)[0]
            dim = tf.shape(z_mean)[1]
            epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
            return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    # Encoder
    encoder_inputs = keras.Input(shape=(128, 128, 4))
    x = layers.Conv2D(32, 3, activation="relu", padding="same", strides=2)(encoder_inputs)
    x = layers.Conv2D(64, 3, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2D(256, 3, activation="relu", padding="same", strides=2)(x)
    x = layers.Flatten()(x)
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling()([z_mean, z_log_var])
    encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

    # Decoder
    latent_inputs = keras.Input(shape=(latent_dim,))
    x = layers.Dense(8 * 8 * 256, activation="relu")(latent_inputs)
    x = layers.Reshape((8, 8, 256))(x)
    x = layers.Conv2DTranspose(128, 4, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2DTranspose(64, 4, activation="relu", padding="same", strides=2)(x)
    x = layers.Conv2DTranspose(32, 4, activation="relu", padding="same", strides=2)(x)
    decoder_outputs = layers.Conv2DTranspose(4, 4, activation="sigmoid", padding="same", strides=2)(x)
    decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")

    # 2. PatchGAN Discriminator
    disc_inputs = keras.Input(shape=(128, 128, 4))
    x = layers.Conv2D(64, 4, strides=2, padding="same")(disc_inputs)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(128, 4, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(256, 4, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(512, 4, strides=1, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    disc_outputs = layers.Conv2D(1, 4, strides=1, padding="same")(x) # Output is a patch grid
    discriminator = keras.Model(disc_inputs, disc_outputs, name="discriminator")

encoder.summary()
decoder.summary()
discriminator.summary()


## 3. VAE-GAN Training Loop

In [ ]:
with strategy.scope():
    class VAEGAN(keras.Model):
        def __init__(self, encoder, decoder, discriminator, beta=0.001, adv_weight=0.05, **kwargs):
            super().__init__(**kwargs)
            self.encoder = encoder
            self.decoder = decoder
            self.discriminator = discriminator
            self.beta = beta
            self.adv_weight = adv_weight
            
            self.loss_fn = keras.losses.BinaryCrossentropy(from_logits=True)
            
            self.g_loss_tracker = keras.metrics.Mean(name="g_loss")
            self.recon_loss_tracker = keras.metrics.Mean(name="recon_loss")
            self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
            self.adv_loss_tracker = keras.metrics.Mean(name="adv_loss")
            self.d_loss_tracker = keras.metrics.Mean(name="d_loss")

        def compile(self, d_optimizer, g_optimizer, **kwargs):
            super().compile(**kwargs)
            self.d_optimizer = d_optimizer
            self.g_optimizer = g_optimizer

        @property
        def metrics(self):
            return [
                self.g_loss_tracker, self.recon_loss_tracker, 
                self.kl_loss_tracker, self.adv_loss_tracker, self.d_loss_tracker
            ]

        def call(self, inputs):
            _, _, z = self.encoder(inputs)
            return self.decoder(z)

        def train_step(self, data):
            x, _ = data

            # 1. Train Discriminator
            with tf.GradientTape() as d_tape:
                _, _, z = self.encoder(x, training=True)
                reconstruction = self.decoder(z, training=True)

                real_output = self.discriminator(x, training=True)
                fake_output = self.discriminator(reconstruction, training=True)

                d_real_loss = self.loss_fn(tf.ones_like(real_output), real_output)
                d_fake_loss = self.loss_fn(tf.zeros_like(fake_output), fake_output)
                d_loss = d_real_loss + d_fake_loss

            d_grads = d_tape.gradient(d_loss, self.discriminator.trainable_weights)
            self.d_optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_weights))

            # 2. Train Generator (VAE Encoder & Decoder)
            with tf.GradientTape() as g_tape:
                z_mean, z_log_var, z = self.encoder(x, training=True)
                reconstruction = self.decoder(z, training=True)
                fake_output = self.discriminator(reconstruction, training=True)

                # Core VAE losses
                recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(x - reconstruction), axis=(1, 2, 3)))
                kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
                kl_loss = self.beta * tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
                
                # Adversarial loss (fool discriminator)
                adv_loss = self.loss_fn(tf.ones_like(fake_output), fake_output)
                
                # Total Generator loss
                g_loss = recon_loss + kl_loss + self.adv_weight * adv_loss

            g_grads = g_tape.gradient(g_loss, self.encoder.trainable_weights + self.decoder.trainable_weights)
            self.g_optimizer.apply_gradients(zip(g_grads, self.encoder.trainable_weights + self.decoder.trainable_weights))

            # Update metrics
            self.g_loss_tracker.update_state(g_loss)
            self.recon_loss_tracker.update_state(recon_loss)
            self.kl_loss_tracker.update_state(kl_loss)
            self.adv_loss_tracker.update_state(self.adv_weight * adv_loss)
            self.d_loss_tracker.update_state(d_loss)

            return {m.name: m.result() for m in self.metrics}
        
        def test_step(self, data):
            # Validation does not train, just calculates losses
            x, _ = data
            z_mean, z_log_var, z = self.encoder(x, training=False)
            reconstruction = self.decoder(z, training=False)
            
            real_output = self.discriminator(x, training=False)
            fake_output = self.discriminator(reconstruction, training=False)

            d_real_loss = self.loss_fn(tf.ones_like(real_output), real_output)
            d_fake_loss = self.loss_fn(tf.zeros_like(fake_output), fake_output)
            d_loss = d_real_loss + d_fake_loss
            
            recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(x - reconstruction), axis=(1, 2, 3)))
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = self.beta * tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            adv_loss = self.loss_fn(tf.ones_like(fake_output), fake_output)
            g_loss = recon_loss + kl_loss + self.adv_weight * adv_loss

            self.g_loss_tracker.update_state(g_loss)
            self.recon_loss_tracker.update_state(recon_loss)
            self.kl_loss_tracker.update_state(kl_loss)
            self.adv_loss_tracker.update_state(self.adv_weight * adv_loss)
            self.d_loss_tracker.update_state(d_loss)

            return {m.name: m.result() for m in self.metrics}

    vaegan = VAEGAN(encoder, decoder, discriminator, beta=0.001, adv_weight=0.01)
    vaegan.compile(
        d_optimizer=keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5),
        g_optimizer=keras.optimizers.Adam(learning_rate=1e-3)
    )


## 4. Execution

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_g_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(DIRS["checkpoints"], "latest.weights.h5"), save_weights_only=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(DIRS["checkpoints"], "best.weights.h5"), save_best_only=True, monitor="val_g_loss", save_weights_only=True),
    keras.callbacks.CSVLogger(os.path.join(DIRS["logs"], "training_log.csv"), append=True)
]

vaegan.build((None, 128, 128, 4))

EPOCHS = 50
STEPS_PER_EPOCH = TRAIN_SIZE // BATCH_SIZE
VAL_STEPS = VAL_SIZE // BATCH_SIZE

# Resumable training logic
initial_epoch = 0
latest_checkpoint = os.path.join(DIRS["checkpoints"], "latest.weights.h5")
log_path = os.path.join(DIRS["logs"], "training_log.csv")

if os.path.exists(latest_checkpoint):
    try:
        print(f"Found checkpoint {latest_checkpoint}, resuming...")
        vaegan.load_weights(latest_checkpoint)
        if os.path.exists(log_path):
            df = pd.read_csv(log_path)
            if len(df) > 0 and 'epoch' in df.columns:
                initial_epoch = int(df['epoch'].iloc[-1]) + 1
    except Exception as e:
        print(f"Could not resume: {e}")

history = vaegan.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VAL_STEPS,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    callbacks=callbacks
)

# Save Full Models & Weights
print("Saving models...")
encoder.save(os.path.join(BASE_DIR, "encoder_vaegan_10k.keras"))
decoder.save(os.path.join(BASE_DIR, "decoder_vaegan_10k.keras"))
discriminator.save(os.path.join(BASE_DIR, "discriminator_vaegan_10k.keras"))
encoder.save_weights(os.path.join(BASE_DIR, "encoder_vaegan_10k.weights.h5"))
decoder.save_weights(os.path.join(BASE_DIR, "decoder_vaegan_10k.weights.h5"))
print("Models saved.")


## 5. Evaluation & Generation

In [ ]:
hist = pd.read_csv(os.path.join(DIRS["logs"], "training_log.csv"))

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.plot(hist['g_loss'], label='Train G Loss')
plt.plot(hist['val_g_loss'], label='Val G Loss')
plt.title('Total Generator Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(hist['recon_loss'], label='Train Recon Loss')
plt.plot(hist['val_recon_loss'], label='Val Recon Loss')
plt.title('Reconstruction Loss (MSE)')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(hist['d_loss'], label='Train D Loss', color='red')
plt.plot(hist['val_d_loss'], label='Val D Loss', color='darkred')
plt.title('Discriminator Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "losses_plot.png"))
plt.show()


In [ ]:
random_indices = random.sample(range(len(test_originals)), 3)
test_reconstructions = vaegan.predict(test_originals)

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for i, idx in enumerate(random_indices):
    axes[0, i].imshow(test_originals[idx])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")
    axes[1, i].imshow(test_reconstructions[idx])
    axes[1, i].set_title("VAE-GAN Reconstructed")
    axes[1, i].axis("off")

plt.suptitle("Test Set Reconstructions", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "reconstructions_plot.png"))
plt.show()


## 6. Generalization Check (Prior-Scaled Variant Generation)

In [ ]:
print("Running Generalization Check on 12 Random Test Sprites...")
random.seed(42)
random_test_indices = random.sample(range(len(test_originals)), 12)
selected_sprites = test_originals[random_test_indices]

z_means, _, _ = encoder.predict(selected_sprites, batch_size=12, verbose=0)

scale_gen = 1.2
tf.random.set_seed(42)
epsilons = tf.random.normal(shape=tf.shape(z_means))
z_variants_gen = z_means + (scale_gen * epsilons)

decoded_variants = decoder.predict(z_variants_gen, batch_size=12, verbose=0)
decoded_recons = decoder.predict(z_means, batch_size=12, verbose=0)

fig, axes = plt.subplots(12, 3, figsize=(10, 36))
for i in range(12):
    axes[i, 0].imshow(selected_sprites[i])
    axes[i, 0].axis("off")
    if i == 0: axes[i, 0].set_title("Original", weight='bold')
    
    axes[i, 1].imshow(decoded_recons[i])
    axes[i, 1].axis("off")
    if i == 0: axes[i, 1].set_title("VAE-GAN Recon", weight='bold')
    
    axes[i, 2].imshow(decoded_variants[i])
    axes[i, 2].axis("off")
    if i == 0: axes[i, 2].set_title(f"Variant (Scale = {scale_gen})", weight='bold')

plt.suptitle(f"Generalization Check: 12 Random Sprites vs Prior-Scaled Variants", fontsize=16, weight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.96)
plt.savefig(os.path.join(BASE_DIR, "variants_comparison_10k.png"), bbox_inches='tight')
plt.show()

vae_mses = [float(np.mean(np.square(selected_sprites[i] - decoded_recons[i]))) for i in range(12)]
variant_mses = [float(np.mean(np.square(decoded_recons[i] - decoded_variants[i]))) for i in range(12)]

results = {
    "random_test_indices": random_test_indices,
    "vae_reconstruction_mses": vae_mses,
    "vae_variant_mses": variant_mses,
    "average_vae_reconstruction_mse": float(np.mean(vae_mses)),
    "average_vae_variant_mse": float(np.mean(variant_mses)),
    "ratio_variant_to_recon": float(np.mean(variant_mses)/np.mean(vae_mses))
}
with open(os.path.join(BASE_DIR, "vaegan_evaluation_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print("Saved evaluation metrics to JSON.")


## 7. Package Outputs for Download

In [ ]:
import shutil
from IPython.display import FileLink

zip_path = "/kaggle/working/VAE_GAN_10K_Outputs"
shutil.make_archive(zip_path, 'zip', BASE_DIR)

print(f"Created {zip_path}.zip!")
print("Click the link below to download your files:")
FileLink(r'VAE_GAN_10K_Outputs.zip')
